In [1]:
import ROOT
import os

ROOT.gROOT.SetBatch(True)
ROOT.gErrorIgnoreLevel = ROOT.kError
ROOT.gStyle.SetOptStat(0)  # 통계창 끄기 (원하면 주석 처리)

# ─── 0) 입력 파일 & 출력 디렉토리 ──────────────────────────────────────
input_samples = [
    ("hist_ST.root",   "ST_s"),
    ("hist_ST_t.root", "ST_t"),
    ("hist_ST_tW.root","ST_tW"),
    ("hist_TT.root",   "TT"),
]

out_dir = "2D_compare_ST_like_overlay_norm_2D"
os.makedirs(out_dir, exist_ok=True)

# 2D 그림 저장 폴더
plot2d_dir = os.path.join(out_dir, "2D_plots_colz")
os.makedirs(plot2d_dir, exist_ok=True)

# ─── 1) 'plots' 디렉토리 안에서 2D 히스토그램 이름만 가져오기 ────────
def get_hist2d_names_from_root(root_file):
    f = ROOT.TFile.Open(root_file)
    if not f or f.IsZombie():
        raise RuntimeError(f"Cannot open file: {root_file}")
    plots_dir = f.Get("plots")
    if not plots_dir:
        raise RuntimeError(f"'plots' directory not found in {root_file}")

    names = []
    for key in plots_dir.GetListOfKeys():
        obj = key.ReadObj()
        # TH2 계열이면서 dimension=2 인 것만 (TH2F, TH2D, TH2I 등 모두 포함)
        if obj.InheritsFrom("TH2") and obj.GetDimension() == 2:
            names.append(obj.GetName())

    f.Close()
    return names

# 첫 번째 파일 기준으로 2D 히스토그램 이름 리스트 얻기
hist2d_names = get_hist2d_names_from_root(input_samples[0][0])
print(f"Found {len(hist2d_names)} 2D histograms in 'plots/':")
for n in hist2d_names:
    print("  ", n)

# ─── 2) 각 파일에서 해당 2D 히스토그램을 COLZ로 그림 & 저장 ────────
# Canvas는 재사용해도 되고, 매번 새로 만들어도 됨. 여기선 재사용.
canvas = ROOT.TCanvas("c2d", "c2d", 800, 700)

for root_file, label in input_samples:
    print(f"\n[INFO] Processing file: {root_file} (label={label})")

    f = ROOT.TFile.Open(root_file)
    if not f or f.IsZombie():
        print(f"  [WARNING] Cannot open file: {root_file}, skip.")
        continue

    plots_dir = f.Get("plots")
    if not plots_dir:
        print(f"  [WARNING] 'plots' directory not found in {root_file}, skip.")
        f.Close()
        continue

    # 파일 이름(확장자 제거) - output 파일명에 사용할 것
    file_base = os.path.splitext(os.path.basename(root_file))[0]

    for hname in hist2d_names:
        # plots/ 디렉토리에서 히스토그램 가져오기
        hobj = plots_dir.Get(hname)
        if not hobj:
            print(f"    [WARNING] Histogram '{hname}' not found in {root_file}, skip.")
            continue
        if not (hobj.InheritsFrom("TH2") and hobj.GetDimension() == 2):
            print(f"    [WARNING] Object '{hname}' is not 2D TH2 in {root_file}, skip.")
            continue

        hist2d = hobj  # 이름만 바꿔서 쓰기

        # 그림 그리기
        canvas.cd()
        canvas.Clear()

        # 축 라벨/타이틀 옵션 조금 조정 (원하면 수정)
        hist2d.SetTitle(f"{file_base} : {hname}")
        # hist2d.GetXaxis().SetTitle(hist2d.GetXaxis().GetTitle())
        # hist2d.GetYaxis().SetTitle(hist2d.GetYaxis().GetTitle())

        hist2d.Draw("COLZ")

        # 출력 파일 이름: 파일베이스__히스토그램이름.png
        # ROOT hist 이름에 '/' 같은게 들어가 있으면 파일 시스템에서 문제될 수 있으니 치환
        safe_hname = hname.replace("/", "_")
        out_name = f"{file_base}__{safe_hname}.png"
        out_path = os.path.join(plot2d_dir, out_name)

        canvas.SaveAs(out_path)
        print(f"    [SAVED] {out_path}")

    f.Close()

print("\n[DONE] All 2D COLZ plots have been created in:", plot2d_dir)


Found 5 2D histograms in 'plots/':
   top_pt_vs_eta
   b_pt_vs_eta
   lep_pt_vs_eta
   b_eta_vs_lep_eta
   b_pt_vs_lep_pt

[INFO] Processing file: hist_ST.root (label=ST_s)
    [SAVED] 2D_compare_ST_like_overlay_norm_2D/2D_plots_colz/hist_ST__top_pt_vs_eta.png
    [SAVED] 2D_compare_ST_like_overlay_norm_2D/2D_plots_colz/hist_ST__b_pt_vs_eta.png
    [SAVED] 2D_compare_ST_like_overlay_norm_2D/2D_plots_colz/hist_ST__lep_pt_vs_eta.png
    [SAVED] 2D_compare_ST_like_overlay_norm_2D/2D_plots_colz/hist_ST__b_eta_vs_lep_eta.png
    [SAVED] 2D_compare_ST_like_overlay_norm_2D/2D_plots_colz/hist_ST__b_pt_vs_lep_pt.png

[INFO] Processing file: hist_ST_t.root (label=ST_t)
    [SAVED] 2D_compare_ST_like_overlay_norm_2D/2D_plots_colz/hist_ST_t__top_pt_vs_eta.png
    [SAVED] 2D_compare_ST_like_overlay_norm_2D/2D_plots_colz/hist_ST_t__b_pt_vs_eta.png
    [SAVED] 2D_compare_ST_like_overlay_norm_2D/2D_plots_colz/hist_ST_t__lep_pt_vs_eta.png
    [SAVED] 2D_compare_ST_like_overlay_norm_2D/2D_plots_colz/his